# 03 Support Vector Machine (SVM) with TF-IDF

Notebook to train and evaluate the SVM model using TF-IDF features

## 1 Import libraries

In [1]:
import pandas as pd
import nltk
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer # Replaces spaCy
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    classification_report
)

nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 2 Load data

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
test_labels = pd.read_csv('data/test_labels.csv')
categories = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

## 3 Feature Extraction (TF-IDF)

replace the spaCy embedding function with a vectorized TF-IDF approach

In [4]:
# Initialize TF-IDF Vectorizer
# ngram_range=(1,2) allows the model to see phrases like "go die" or "shut up"
tfidf = TfidfVectorizer(
    max_features=30000, 
    stop_words='english', 
    sublinear_tf=True, 
    ngram_range=(1, 2)
)

In [5]:
X_train = tfidf.fit_transform(train['comment_text'].astype(str))
y_train = train[categories].values

## 4 Build and Train the Model

In [ ]:
svm_model = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=3000))
svm_model.fit(X_train, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LinearSVC(cla...max_iter=3000)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower 

## 5 Predictions on Test Set

In [7]:
test_combined = pd.merge(test, test_labels, on='id')
test_scored = test_combined[test_combined['toxic'] != -1].copy()

In [ ]:
# Transform test data
X_test = tfidf.transform(test_scored['comment_text'].astype(str))
y_test = test_scored[categories].values

# Predictions
y_pred = svm_model.predict(X_test)
y_score = svm_model.decision_function(X_test)

## 6 Metrics

In [9]:
metrics_list = []

for i, category in enumerate(categories):
    acc = accuracy_score(y_test[:, i], y_pred[:, i])
    prec = precision_score(y_test[:, i], y_pred[:, i], zero_division=0)
    rec = recall_score(y_test[:, i], y_pred[:, i], zero_division=0)
    f1 = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    auc = roc_auc_score(y_test[:, i], y_score[:, i])
    
    metrics_list.append({
        'Category': category,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'AUC': auc
    })

results_df = pd.DataFrame(metrics_list)

In [10]:
results_df.round(4)

,Category,Accuracy,Precision,Recall,F1-Score,AUC
0,toxic,0.8812,0.4390,0.8918,0.5884,0.9520
1,severe_toxic,0.9724,0.1423,0.7575,0.2396,0.9693
2,obscene,0.9359,0.4688,0.8372,0.6011,0.9641
3,threat,0.9939,0.3075,0.6777,0.4231,0.9781
4,insult,0.9256,0.4021,0.7984,0.5348,0.9509
5,identity_hate,0.9732,0.2484,0.6938,0.3658,0.9610


In [11]:
# Calculate Global (Macro) Metrics
macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
macro_prec = precision_score(y_test, y_pred, average='macro', zero_division=0)
macro_rec = recall_score(y_test, y_pred, average='macro', zero_division=0)
macro_auc = roc_auc_score(y_test, y_score, average='macro')
subset_accuracy = accuracy_score(y_test, y_pred)

In [12]:
print("Summary Global Metrics (Macro Average) ---")
print(f"Exact Match Accuracy: {subset_accuracy:.4f}")
print(f"Macro Precision:      {macro_prec:.4f}")
print(f"Macro Recall:         {macro_rec:.4f}")
print(f"Macro F1-Score:       {macro_f1:.4f}")
print(f"Macro AUC:            {macro_auc:.4f}")

Summary Global Metrics (Macro Average) ---
Exact Match Accuracy: 0.8066
Macro Precision:      0.3347
Macro Recall:         0.7761
Macro F1-Score:       0.4588
Macro AUC:            0.9625


End of notebook